## Working with bigger data – online algorithms and out-of-core learning

In [43]:
import numpy as np
import re
from nltk.corpus import stopwords

stop = stopwords.words("english")


def tokenizer(text):
    text = re.sub("<[^>]*>", "", text)
    emoticons = re.findall("(?::|;|=)(?:-)?(?:\)|\(|D|P)", text)
    text = re.sub("[\W]+", " ", text.lower()) + " ".join(emoticons).replace("-", "")
    tokenized = [w for w in text.split() if w not in stop]
    return tokenized


<>:10: SyntaxWarning: invalid escape sequence '\)'
<>:11: SyntaxWarning: invalid escape sequence '\W'
<>:10: SyntaxWarning: invalid escape sequence '\)'
<>:11: SyntaxWarning: invalid escape sequence '\W'
/tmp/ipykernel_210166/2441316538.py:10: SyntaxWarning: invalid escape sequence '\)'
  emoticons = re.findall("(?::|;|=)(?:-)?(?:\)|\(|D|P)", text)
/tmp/ipykernel_210166/2441316538.py:11: SyntaxWarning: invalid escape sequence '\W'
  text = re.sub("[\W]+", " ", text.lower()) + " ".join(emoticons).replace("-", "")


In [44]:
# Generator function
def stream_docs(path):
    with open(path, "r", encoding="utf-8") as csv:
        next(csv)  # skip header
        for line in csv:
            text, label = line[:-3], int(line[-2])
            yield text, label


next(stream_docs(path="movie_data.csv"))

('"In 1974, the teenager Martha Moxley (Maggie Grace) moves to the high-class area of Belle Haven, Greenwich, Connecticut. On the Mischief Night, eve of Halloween, she was murdered in the backyard of her house and her murder remained unsolved. Twenty-two years later, the writer Mark Fuhrman (Christopher Meloni), who is a former LA detective that has fallen in disgrace for perjury in O.J. Simpson trial and moved to Idaho, decides to investigate the case with his partner Stephen Weeks (Andrew Mitchell) with the purpose of writing a book. The locals squirm and do not welcome them, but with the support of the retired detective Steve Carroll (Robert Forster) that was in charge of the investigation in the 70\'s, they discover the criminal and a net of power and money to cover the murder.<br /><br />""Murder in Greenwich"" is a good TV movie, with the true story of a murder of a fifteen years old girl that was committed by a wealthy teenager whose mother was a Kennedy. The powerful and rich f

In [45]:
def get_minibatch(doc_stream, size: int):
    docs, y = [], []
    try:
        for _ in range(size):
            text, label = next(doc_stream)
            docs.append(text)
            y.append(label)
    except StopIteration:
        return None, None

    return docs, y

In [46]:
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier

vect = HashingVectorizer(
    decode_error="ignore", n_features=2 * 21, preprocessor=None, tokenizer=tokenizer
)

clf = SGDClassifier(loss="log_loss", random_state=1)
doc_stream = stream_docs(path="movie_data.csv")

---
#### Start out-of-core learning:

In [47]:
import pyprind

BATCH_SIZE = 45
pbar = pyprind.ProgBar(BATCH_SIZE)
classes = np.array([0, 1])

for _ in range(BATCH_SIZE):
    X_train, y_train = get_minibatch(doc_stream, size=1000)
    if not X_train:
        break
    X_train = vect.transform(X_train)
    clf.partial_fit(X_train, y_train, classes=classes)  # type: ignore
    pbar.update()

In [48]:
X_test, y_test = get_minibatch(doc_stream, size=5000)
X_test = vect.transform(X_test)
clf = clf.partial_fit(X_test, y_test)
print(f"Accuracy: {clf.score(X_test, y_test):.3f}")

Accuracy: 0.621
